In [18]:
!pip install -q transformers torch sentencepiece language-tool-python nltk

In [19]:
import torch
print(torch.__version__)

2.11.0+cpu


In [20]:
import nltk

In [21]:
import language_tool_python

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [23]:
print("Libraries imported successfully!")
print("PyTorch version:", torch.__version__)

Libraries imported successfully!
PyTorch version: 2.11.0+cpu


In [24]:
#Download NLTK resources

nltk.download("punkt")
nltk.download("punkt_tab")

print("NLTK resources downloaded successfully!")

NLTK resources downloaded successfully!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [25]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Hugging Face settings applied")

Hugging Face settings applied


In [26]:
#Load the T5 paraphrasing model

from huggingface_hub import snapshot_download

model_name = "Vamsi/T5_Paraphrase_Paws"

print("Downloading model files...")

model_path = snapshot_download(
    repo_id=model_name,
    allow_patterns=[
        "config.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "spiece.model",
        "tokenizer.json",
        "pytorch_model.bin"
    ]
)

print("Model downloaded successfully!")
print("Model path:", model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Model downloaded successfully!
Model path: /root/.cache/huggingface/hub/models--Vamsi--T5_Paraphrase_Paws/snapshots/ac7aaff59cc52a0283124eb04004f918c16b7420


Load tokenizer and model

In [27]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path,
    use_safetensors=False
)

print("T5 model loaded successfully!")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

T5 model loaded successfully!


Create paraphrasing function

In [28]:
def paraphrase_text(text, num_return_sequences=3):

    input_text = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = model.generate(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"],
            max_length=128,
            num_beams=10,
            num_return_sequences=num_return_sequences,
            temperature=1.5,
            do_sample=True,
            early_stopping=True
        )

    paraphrases = []

    for output in outputs:
        result = tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        paraphrases.append(result)

    return paraphrases

In [29]:
#Test the paraphrasing model


input_text = "Artificial intelligence is changing the way people work."

paraphrased_results = paraphrase_text(input_text)

print("Original Text:")
print(input_text)

print("\nParaphrased Texts:")

for i, result in enumerate(paraphrased_results, 1):
    print(f"{i}. {result}")

Original Text:
Artificial intelligence is changing the way people work.

Paraphrased Texts:
1. Artificial intelligence is changing the way people work.
2. Artificial intelligence is changing the way people work .
3. Artificial Intelligence is changing the way people work.


Grammar and spelling checker

In [30]:
tool = language_tool_python.LanguageTool("en-US")

def grammar_check(text):

    matches = tool.check(text)

    return matches

In [31]:
#Create quality checking function

def check_quality(text):

    matches = grammar_check(text)

    grammar_errors = []
    spelling_errors = []

    for match in matches:

        error = {
            "message": match.message,
            "suggestions": match.replacements[:3],
            "context": match.context
        }

        if "spelling" in match.message.lower():
            spelling_errors.append(error)
        else:
            grammar_errors.append(error)

    return grammar_errors, spelling_errors

Check a paraphrased sentence

In [39]:
sample_text = paraphrased_results[0]

grammar_errors, spelling_errors = check_quality(sample_text)

print("Paraphrased Text:")
print(sample_text)

print("\nGrammar Errors:", len(grammar_errors))
print("Spelling Errors:", len(spelling_errors))

Paraphrased Text:
Artificial intelligence is changing the way people work.

Grammar Errors: 0
Spelling Errors: 0


#Display detected errors

if grammar_errors:
    
    print("\nGrammar Issues:")
    
    for error in grammar_errors:
        print("Message:", error["message"])
        print("Context:", error["context"])
        print("Suggestions:", error["suggestions"])
        print()

else:
    print("\nNo grammar errors detected.")


if spelling_errors:
    
    print("\nSpelling Issues:")
    
    for error in spelling_errors:
        print("Message:", error["message"])
        print("Context:", error["context"])
        print("Suggestions:", error["suggestions"])
        print()

else:
    print("No spelling errors detected.")

In [33]:
#Fluency check

def fluency_score(text):

    grammar_errors, spelling_errors = check_quality(text)

    total_errors = len(grammar_errors) + len(spelling_errors)

    words = text.split()

    if len(words) == 0:
        return 0

    score = 100 - (total_errors / len(words) * 100)

    score = max(0, min(100, score))

    return round(score, 2)

Test fluency

In [34]:
for i, result in enumerate(paraphrased_results, 1):

    score = fluency_score(result)

    print(f"Paraphrase {i}:")
    print(result)
    print("Fluency Score:", score, "/ 100")
    print()

Paraphrase 1:
Artificial intelligence is changing the way people work.
Fluency Score: 100 / 100

Paraphrase 2:
Artificial intelligence is changing the way people work .
Fluency Score: 88.89 / 100

Paraphrase 3:
Artificial Intelligence is changing the way people work.
Fluency Score: 100 / 100



In [50]:
#originality evaluation

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference_text, candidate_text):
    reference = reference_text.split()
    candidate = candidate_text.split()

    smoothing = SmoothingFunction().method1

    score = sentence_bleu(
        [reference],
        candidate,
        smoothing_function=smoothing
    )

    return round(score, 4)

In [51]:
#Evaluate all paraphrases


print("Evaluation Results\n")

for i, result in enumerate(paraphrased_results, 1):

    bleu_score = calculate_bleu(input_text, result)
    fluency = fluency_score(result)

    print(f"Paraphrase {i}")
    print("-" * 50)
    print("Output:", result)
    print("BLEU Score:", bleu_score)
    print("Fluency Score:", fluency, "/ 100")
    print()

Evaluation Results

Paraphrase 1
--------------------------------------------------
Output: Artificial intelligence is changing the way people work.
BLEU Score: 1.0
Fluency Score: 100 / 100

Paraphrase 2
--------------------------------------------------
Output: Artificial intelligence is changing the way people work .
BLEU Score: 0.726
Fluency Score: 88.89 / 100

Paraphrase 3
--------------------------------------------------
Output: Artificial Intelligence is changing the way people work.
BLEU Score: 0.7071
Fluency Score: 100 / 100



In [52]:
#Build the complete tool

def paraphrasing_tool(text):

    print("=" * 60)
    print("AI-POWERED PARAPHRASING TOOL")
    print("=" * 60)

    print("\nOriginal Text:")
    print(text)

    results = paraphrase_text(text, num_return_sequences=3)

    print("\nParaphrased Results:\n")

    for i, result in enumerate(results, 1):

        grammar_errors, spelling_errors = check_quality(result)

        fluency = fluency_score(result)
        bleu = calculate_bleu(text, result)

        print(f"Result {i}")
        print("-" * 50)
        print("Paraphrased Text:")
        print(result)

        print("\nGrammar Errors:", len(grammar_errors))
        print("Spelling Errors:", len(spelling_errors))
        print("Fluency Score:", fluency, "/ 100")
        print("BLEU Score:", bleu)

        print()

    print("=" * 60)

In [53]:
#checking


text = """
Machine learning is becoming an important technology in many industries.
"""

paraphrasing_tool(text)

AI-POWERED PARAPHRASING TOOL

Original Text:

Machine learning is becoming an important technology in many industries.


Paraphrased Results:

Result 1
--------------------------------------------------
Paraphrased Text:
Machine learning is becoming an important technology in many industries .

Grammar Errors: 1
Spelling Errors: 0
Fluency Score: 90.91 / 100
BLEU Score: 0.7861

Result 2
--------------------------------------------------
Paraphrased Text:
Machine learning is becoming an important technology in many industries.

Grammar Errors: 0
Spelling Errors: 0
Fluency Score: 100 / 100
BLEU Score: 1.0

Result 3
--------------------------------------------------
Paraphrased Text:
In many industries, machine learning is becoming an important technology .

Grammar Errors: 1
Spelling Errors: 0
Fluency Score: 90.91 / 100
BLEU Score: 0.4799



In [47]:
#Test with another example

text2 = """
Students can use artificial intelligence tools to improve their learning experience.
"""

paraphrasing_tool(text2)

AI-POWERED PARAPHRASING TOOL

Original Text:

Students can use artificial intelligence tools to improve their learning experience.


Paraphrased Results:

Result 1
--------------------------------------------------
Paraphrased Text:
Students can use artificial intelligence tools to improve their learning experience .

Grammar Errors: 1
Spelling Errors: 0
Fluency Score: 91.67 / 100
BLEU Score: 0.8071

Result 2
--------------------------------------------------
Paraphrased Text:
Students can use artificial intelligence tools to improve their learning experience.

Grammar Errors: 0
Spelling Errors: 0
Fluency Score: 100 / 100
BLEU Score: 1.0

Result 3
--------------------------------------------------
Paraphrased Text:
Students can use artificial intelligence tools to enhance their learning experience .

Grammar Errors: 1
Spelling Errors: 0
Fluency Score: 91.67 / 100
BLEU Score: 0.5707



In [49]:
#User input version

user_text = input("Enter the text you want to paraphrase: ")

paraphrasing_tool(user_text)

Enter the text you want to paraphrase: i went a forest it was beatufil
AI-POWERED PARAPHRASING TOOL

Original Text:
i went a forest it was beatufil

Paraphrased Results:

Result 1
--------------------------------------------------
Paraphrased Text:
I went to a forest it was beatufil .

Grammar Errors: 1
Spelling Errors: 1
Fluency Score: 77.78 / 100
BLEU Score: 0.5133

Result 2
--------------------------------------------------
Paraphrased Text:
I went a forest it was beatufil

Grammar Errors: 0
Spelling Errors: 1
Fluency Score: 85.71 / 100
BLEU Score: 1.0

Result 3
--------------------------------------------------
Paraphrased Text:
i went a forest it was beatufil .

Grammar Errors: 2
Spelling Errors: 1
Fluency Score: 62.5 / 100
BLEU Score: 0.8409



Conclusion
----------
This project developed an AI-powered paraphrasing tool using a pre-trained T5 Transformer model from Hugging Face. The tool accepts input text and generates multiple paraphrased versions while trying to preserve the original meaning. Grammar and spelling checks were added to examine the quality of the generated text. BLEU scores were also used to compare the paraphrased text with the original text. The project demonstrates how Transformer-based deep learning models can be used for natural language processing and text generation.